In [4]:
#Loading in general twitter dataset --> additional user info 
import pandas as pd
import contractions
import re
import emoji
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
import matplotlib.pyplot as plt
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
import gensim
from gensim.models import Word2Vec
from sklearn.decomposition import LatentDirichletAllocation
import pyLDAvis
#import pyLDAvis.sklearn
import pyLDAvis.lda_model
from langdetect import detect
general = pd.read_csv('general_Twitter.csv', encoding='latin1')
print(general.head())

    _unit_id  _golden _unit_state  _trusted_judgments _last_judgment_at  \
0  815719226    False   finalized                   3    10/26/15 23:24   
1  815719227    False   finalized                   3    10/26/15 23:30   
2  815719228    False   finalized                   3    10/26/15 23:33   
3  815719229    False   finalized                   3    10/26/15 23:10   
4  815719230    False   finalized                   3     10/27/15 1:15   

   gender  gender:confidence profile_yn  profile_yn:confidence  \
0    male             1.0000        yes                    1.0   
1    male             1.0000        yes                    1.0   
2    male             0.6625        yes                    1.0   
3    male             1.0000        yes                    1.0   
4  female             1.0000        yes                    1.0   

            created  ...                                       profileimage  \
0  12/05/2013 01:48  ...  https://pbs.twimg.com/profile_images/414342229.

In [5]:
general.describe()
general.info()

##drop the following columns 
##golden: whether the user was included in the gold standard for the model; TRUE or FALSE
##unit_state: state of the observation; one of finalized (for contributor-judged) or golden (for gold standard observations)
##trusted_judgments: number of trusted judgments (int); always 3 for non-golden, and what may be a unique id for gold standard observations
##last_judgment_at: date and time of last contributor judgment; blank for gold standard observations
##profile_yn: "no" here seems to mean that the profile was meant to be part of the dataset but was not available when contributors went to judge it
##profile_yn:confidence: confidence in the existence/non-existence of the profile
##gender_gold: if the profile is golden, what is the gender?
##profile_yn_gold: whether the profile y/n value is golden

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20050 entries, 0 to 20049
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   _unit_id               20050 non-null  int64  
 1   _golden                20050 non-null  bool   
 2   _unit_state            20050 non-null  object 
 3   _trusted_judgments     20050 non-null  int64  
 4   _last_judgment_at      20000 non-null  object 
 5   gender                 19953 non-null  object 
 6   gender:confidence      20024 non-null  float64
 7   profile_yn             20050 non-null  object 
 8   profile_yn:confidence  20050 non-null  float64
 9   created                20050 non-null  object 
 10  description            16306 non-null  object 
 11  fav_number             20050 non-null  int64  
 12  gender_gold            50 non-null     object 
 13  link_color             20050 non-null  object 
 14  name                   20050 non-null  object 
 15  pr

In [6]:
general = general.drop(['_golden','_unit_state','_trusted_judgments','_last_judgment_at','profile_yn','profile_yn:confidence','gender_gold','profile_yn_gold'],axis=1)
print(general.head())
#tfidf = tfidf_tweets.drop('unit_id', axis=1)

    _unit_id  gender  gender:confidence           created  \
0  815719226    male             1.0000  12/05/2013 01:48   
1  815719227    male             1.0000  10/01/2012 13:51   
2  815719228    male             0.6625    11/28/14 11:30   
3  815719229    male             1.0000  06/11/2009 22:39   
4  815719230  female             1.0000     4/16/14 13:23   

                                         description  fav_number link_color  \
0                              i sing my own rhythm.           0     08C2C2   
1  I'm the author of novels filled with family dr...          68     0084B4   
2                louis whining and squealing and all        7696     ABB8C2   
3  Mobile guy.  49ers, Shazam, Google, Kleiner Pe...         202     0084B4   
4  Ricky Wilson The Best FRONTMAN/Kaiser Chiefs T...       37318     3B94D9   

             name                                       profileimage  \
0         sheezy0  https://pbs.twimg.com/profile_images/414342229...   
1     DavdBurn

In [7]:
##identifying and dropping null values
null_values = general[general['text'].isnull()]
#print(null_values)
##dropping entries where gender is null - 97
null_Gender_values = general[general['gender'].isnull()]
print(len(null_Gender_values))
#general = general.dropna(subset=['de'])

##null values expected for tweet_coord and tweet location


97


In [8]:
general = general.dropna(subset=['gender','description'], axis=0)
print(general['gender'].describe)

<bound method NDFrame.describe of 0          male
1          male
2          male
3          male
4        female
          ...  
20045    female
20046      male
20047      male
20048    female
20049    female
Name: gender, Length: 16224, dtype: object>


In [9]:
general.isna()

,_unit_id,gender,gender:confidence,created,description,fav_number,link_color,name,profileimage,retweet_count,sidebar_color,text,tweet_coord,tweet_count,tweet_created,tweet_id,tweet_location,user_timezone
0,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False
2,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20045,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True
20046,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True
20047,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True
20048,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True


In [10]:

'''
null_created_values = general[general['created'].isnull()]
print('num of null created values' ,len(null_created_values))
'''

##for location columns : tweet_coord, tweet_location, user_timezone values expected null values as its based on if the user has location settings turned on
##drop the null description values 

"\nnull_created_values = general[general['created'].isnull()]\nprint('num of null created values' ,len(null_created_values))\n"

In [11]:
###checking for duplicate values
duplicates = general[general.duplicated(keep='first')]
print(duplicates)

Empty DataFrame
Columns: [_unit_id, gender, gender:confidence, created, description, fav_number, link_color, name, profileimage, retweet_count, sidebar_color, text, tweet_coord, tweet_count, tweet_created, tweet_id, tweet_location, user_timezone]
Index: []


In [12]:
general['text'].head

<bound method NDFrame.head of 0        Robbie E Responds To Critics After Win Against...
1        ÛÏIt felt like they were my friends and I was...
2        i absolutely adore when louis starts the songs...
3        Hi @JordanSpieth - Looking at the url - do you...
4        Watching Neighbours on Sky+ catching up with t...
                               ...                        
20045    @lookupondeath ...Fine, and I'll drink tea too...
20046    Greg Hardy you a good player and all but don't...
20047    You can miss people and still never want to se...
20048    @bitemyapp i had noticed your tendency to pee ...
20049    I think for my APUSH creative project I'm goin...
Name: text, Length: 16224, dtype: object>

In [13]:
##dropping tweets from brands as doesnt fit into anaylsing peoples behaiviours 
general.drop(general[general['gender'] == 'brand'].index, inplace=True)

In [14]:
###Tweet column cleaning ###
#for the text column specifically as it is longer text more cleaning + preprocessing required, same process as labelled dataset

##removing numbers 
def remove_numbers(tweet):
    return ''.join([char for char in tweet if not char.isdigit()])
general['text'] = general['text'].apply(remove_numbers)

### fixing contractions  
def fix_contractions(tweet):
    return contractions.fix(tweet)
general['text'] = general['text'].apply(fix_contractions)

#### Putting hashtags into a separate column to analyse 
def extract_hashtags(text):
    return list(set(re.findall(r'#(\w+)', text)))

general['hashtags'] = general['text'].apply(extract_hashtags)

### removing punctuation and special characters 
general['text'] = general['text'].str.replace('\W',' ',regex=True)



### removing non english text
def remove_non_english_chars(text):
    return re.sub(r'[^\x00-\x7F]+', '', text)
general['text'] = general['text'].apply(remove_non_english_chars)

def remove_words_with_number(text):
    return re.sub(r'\b\w*\d\w*\b', '',text)
general['text'] = general['text'].apply(remove_words_with_number)
### remove non-alphanumeric characters and extra spaces
def remove_non_alphanumeric(text):
    return re.sub(r'[^a-zA-Z0-9\s]', '', text)
general['text'] = general['text'].apply(remove_non_alphanumeric)

### removing URLs from tweets
#tweets['post_text'] = tweets['post_text'].str.replace('http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', ' ')
general['text'] = general['text'].apply(lambda text: re.sub(r'https?:\/\/\S*', '', text, flags=re.MULTILINE))

def remove_links(text):
    return re.sub(r'(@[A-Za-z0–9_]+)|[^\w\s]|#|http\S+', '',text)
general['text'] = general['text'].apply(remove_links)
### convert text to lower case
general['text'] = general['text'].str.lower()

### removing emojis 
def remove_emojis(text):
    #text repesention of emojis 
    emojis_text_rep = emoji.demojize(text)
    clean_text = re.sub(r':[a-zA-Z_]+:', '', emojis_text_rep)
    return clean_text

general['text'] = general['text'].apply(remove_emojis)
def remove_non_english_words(text):
    try:
        return detect(text)=='en'
    except:
        return False 
general['text']= general['text'].apply(lambda x: '' if not remove_non_english_words(x) else x)

print(general['text'].head)

<>:21: DeprecationWarning: invalid escape sequence '\W'
<>:21: DeprecationWarning: invalid escape sequence '\W'
C:\Users\mayur\AppData\Local\Temp\ipykernel_31560\1909294010.py:21: DeprecationWarning: invalid escape sequence '\W'
  general['text'] = general['text'].str.replace('\W',' ',regex=True)


<bound method NDFrame.head of 0        robbie e responds to critics after win against...
1         it felt like they were my friends and i was l...
2        i absolutely adore when louis starts the songs...
3        hi  jordanspieth   looking at the url   do you...
4        watching neighbours on sky  catching up with t...
                               ...                        
20045     lookupondeath    fine  and i will drink tea t...
20046    greg hardy you a good player and all but do no...
20047    you can miss people and still never want to se...
20048     bitemyapp i had noticed your tendency to pee ...
20049    i think for my apush creative project i am goi...
Name: text, Length: 11896, dtype: object>


Tokenisation

In [15]:
general['tokenised'] = general['text'].apply(lambda text: word_tokenize(text))
general['tokenised'].head

<bound method NDFrame.head of 0        [robbie, e, responds, to, critics, after, win,...
1        [it, felt, like, they, were, my, friends, and,...
2        [i, absolutely, adore, when, louis, starts, th...
3        [hi, jordanspieth, looking, at, the, url, do, ...
4        [watching, neighbours, on, sky, catching, up, ...
                               ...                        
20045    [lookupondeath, fine, and, i, will, drink, tea...
20046    [greg, hardy, you, a, good, player, and, all, ...
20047    [you, can, miss, people, and, still, never, wa...
20048    [bitemyapp, i, had, noticed, your, tendency, t...
20049    [i, think, for, my, apush, creative, project, ...
Name: tokenised, Length: 11896, dtype: object>

In [16]:
stop_words = set(stopwords.words('english'))
general['tokenised'] = general['tokenised'].apply(lambda x: [word for word in x if word not in stop_words])
print(general['tokenised'])

0        [robbie, e, responds, critics, win, eddie, edw...
1        [felt, like, friends, living, story, co, arnge...
2        [absolutely, adore, louis, starts, songs, hits...
3        [hi, jordanspieth, looking, url, use, ifttt, t...
4        [watching, neighbours, sky, catching, neighbs,...
                               ...                        
20045              [lookupondeath, fine, drink, tea, love]
20046    [greg, hardy, good, player, get, face, dez, br...
20047              [miss, people, still, never, want, see]
20048    [bitemyapp, noticed, tendency, pee, carpet, wa...
20049    [think, apush, creative, project, going, bring...
Name: tokenised, Length: 11896, dtype: object


Lemmatisation

In [18]:
##lemmatisation
#chose lemmatisation over stemming as it is more accurate 
#nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
#function to apply lemmatisation to each tweet in post_text column
def lemmatisation(tweet):
    #w_tokeniser = nltk.tokenize.WhitespaceTokenizer()
    tweet = [word for word in tweet if word.strip()]

    lemmatiser = WordNetLemmatizer()
    lemmatised_tweet = [lemmatiser.lemmatize(word) for word in tweet]
    #print(lemmatised_tweet)
    return lemmatised_tweet

def lemmatize_tweets(tweet):
    ##initialise wodnetlemmatiser 
    lemmatizer = WordNetLemmatizer()
    lemmatized_sentence = []
    for word, tag in pos_tag(tweet):
        if tag.startswith('NN'):
            #if n then noun
            pos = 'n'  
        elif tag.startswith('VB'):
            #if vb then verb
            pos = 'v' 
        else:
            ##if not above then adjective
            pos = 'a'  
        lemmatized_sentence.append(lemmatizer.lemmatize(word, pos))
    return lemmatized_sentence

general['lemmatised'] = general['tokenised'].apply(lemmatize_tweets)
print(general['lemmatised'].head)

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\mayur\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


<bound method NDFrame.head of 0        [robbie, e, respond, critic, win, eddie, edwar...
1        [felt, like, friend, live, story, co, arngeyhn...
2        [absolutely, adore, louis, start, song, hit, h...
3        [hi, jordanspieth, look, url, use, ifttt, typi...
4        [watch, neighbour, sky, catch, neighbs, xxx, xxx]
                               ...                        
20045              [lookupondeath, fine, drink, tea, love]
20046    [greg, hardy, good, player, get, face, dez, br...
20047              [miss, people, still, never, want, see]
20048    [bitemyapp, notice, tendency, pee, carpet, wan...
20049    [think, apush, creative, project, go, bring, b...
Name: lemmatised, Length: 11896, dtype: object>


Vectorisation 

In [ ]:
### tf-idf vectorisation

tfidfconverter = TfidfVectorizer(analyzer='word',max_df=0.85)
general['string_tweet'] = general['lemmatised'].apply(lambda tokens: ' '.join(tokens))
print(general['text'].head)
# fit and conveert the 'string_tweet' column
X = tfidfconverter.fit_transform(general['string_tweet'])
print(X)


tfidf_df = pd.DataFrame(X.toarray(), columns=tfidfconverter.get_feature_names_out())
tfidf_df['unit_id'] = general['_unit_id'].reset_index(drop=True)

#print(tfidf_df['label'].head)

print(len(tfidf_df['label']))



<bound method NDFrame.head of 0        robbie e responds to critics after win against...
1         it felt like they were my friends and i was l...
2        i absolutely adore when louis starts the songs...
3        hi  jordanspieth   looking at the url   do you...
4        watching neighbours on sky  catching up with t...
                               ...                        
20045     lookupondeath    fine  and i will drink tea t...
20046    greg hardy you a good player and all but do no...
20047    you can miss people and still never want to se...
20048     bitemyapp i had noticed your tendency to pee ...
20049    i think for my apush creative project i am goi...
Name: text, Length: 11896, dtype: object>
  (0, 14537)	0.39340425665433204
  (0, 3735)	0.09131010963273309
  (0, 23056)	0.39340425665433204
  (0, 5926)	0.3425480337118605
  (0, 5889)	0.35620720748206897
  (0, 22858)	0.22553614624353244
  (0, 4338)	0.3652657752073268
  (0, 17287)	0.3144095522648553
  (0, 17563)	0.3934042

In [ ]:
print(tfidf_df)
tfidf_df.to_csv('General_Dataset_TF-IDF_vectorised_tweets.csv', index = False)

       aaaaa  aaaaaaaa  aaaaahhhhhmmmm  aaallll  aaand  aabramz  aam  aap  \
0        0.0       0.0             0.0      0.0    0.0      0.0  0.0  0.0   
1        0.0       0.0             0.0      0.0    0.0      0.0  0.0  0.0   
2        0.0       0.0             0.0      0.0    0.0      0.0  0.0  0.0   
3        0.0       0.0             0.0      0.0    0.0      0.0  0.0  0.0   
4        0.0       0.0             0.0      0.0    0.0      0.0  0.0  0.0   
...      ...       ...             ...      ...    ...      ...  ...  ...   
11891    0.0       0.0             0.0      0.0    0.0      0.0  0.0  0.0   
11892    0.0       0.0             0.0      0.0    0.0      0.0  0.0  0.0   
11893    0.0       0.0             0.0      0.0    0.0      0.0  0.0  0.0   
11894    0.0       0.0             0.0      0.0    0.0      0.0  0.0  0.0   
11895    0.0       0.0             0.0      0.0    0.0      0.0  0.0  0.0   

       aapl  aaron  ...  zyrszbijgl  zzchzufvo  zzcwczlaqc  zzksitayz  zzlx

Word2Vec vectorisation

In [19]:
##use tfidf aswell and run both through classfier and compare
word2vec_model = Word2Vec(sentences=general['lemmatised'], vector_size=200, window=5, min_count=1)
word2vec_model.save('word2vec_model.model')  # Save the trained model


#a function to get the vector representation for each tweet
def get_vector_for_tweet(model, tweet):
    vector = []
    for word in tweet:
        if word in model.wv:
            vector.append(model.wv[word])
    return vector

def tweet_vectorisation(w2v, tweet):
    vector_sum = np.zeros((w2v.vector_size,), dtype="float32")
    word_count = 0
    for word in tweet:
        if word in w2v.wv:
            vector_sum =vector_sum +  w2v.wv[word]
            word_count = word_count+ 1
    if word_count != 0:
        return vector_sum / word_count
    else:
        return vector_sum
    
#filtered_tweets = pd.DataFrame(filtered_tweets)
general['w2v_vectorized'] = general['lemmatised'].apply(lambda x: tweet_vectorisation(word2vec_model,x))
print(general['w2v_vectorized'].head)

<bound method NDFrame.head of 0        [0.06628149, -0.07270818, 0.05771472, 0.123738...
1        [0.14043772, -0.14695702, 0.117248006, 0.24963...
2        [0.09856349, -0.10199013, 0.07795049, 0.167939...
3        [0.091895655, -0.0938449, 0.07297064, 0.159179...
4        [0.04387572, -0.045992587, 0.03521721, 0.07532...
                               ...                        
20045    [0.072796844, -0.073631786, 0.057709895, 0.116...
20046    [0.07547217, -0.07985609, 0.060597185, 0.13097...
20047    [0.18521786, -0.18972044, 0.14186889, 0.309974...
20048    [0.069768466, -0.07443268, 0.053103123, 0.1170...
20049    [0.07929348, -0.08690071, 0.06643565, 0.143183...
Name: w2v_vectorized, Length: 11896, dtype: object>


In [20]:
general.to_csv('w2v_general_dataset.csv', index = False)